# Chọn sản phẩm theo dõi hằng ngày bằng Pareto

## Mục tiêu
Dùng nguyên tắc Pareto để xác định **những danh mục đóng góp phần lớn tổng lượng bán tích lũy**, sau đó lấy danh sách sản phẩm thuộc các danh mục đó làm phạm vi theo dõi giá hằng ngày.

> `all_time_quantity_sold` là chỉ số bán tích lũy tại thời điểm crawl, không phải số bán riêng trong tháng. Vì vậy, kết quả dùng để chọn nhóm sản phẩm đáng theo dõi, không phải để kết luận doanh thu/lượt bán trong tháng 10.

## Quy tắc xử lý
- Chỉ giữ bản ghi crawl mới nhất cho mỗi `seller_product_id`.
- Nếu một `product_id` xuất hiện ở nhiều gian hàng, dùng giá trị `all_time_quantity_sold` lớn nhất làm chỉ số đại diện để tránh cộng trùng cùng sản phẩm giữa các gian hàng.
- Nếu một `product_id` được gán nhiều `category_id`, bên dưới chỉ giữ một danh mục cho mỗi sản phẩm để tránh đếm sản phẩm nhiều lần. Cần xem đây là quy tắc kỹ thuật; nếu dữ liệu có danh mục chính thức thì nên ưu tiên danh mục chính đó.
- Ngưỡng Pareto 80% được lấy đến hết danh mục đầu tiên làm tỷ trọng lũy kế đạt/vượt 80%.

## 1. Đọc dữ liệu

In [ ]:
import pandas as pd
from pathlib import Path

DATA_DIR = Path(r'D:\TiKi-Analysic-final\TiKi-Analysic\01_craw_data')

categories = pd.read_csv(DATA_DIR / 'categories' / 'categories.csv', encoding='utf-8')
product_id = pd.read_csv(DATA_DIR / 'products' / 'product_id.csv', encoding='utf-8')
product_price_history = pd.read_csv(DATA_DIR / 'products' / 'product_price_history.csv', encoding='utf-8')

print('Danh mục:', categories.shape)
print('Danh sách sản phẩm/gian hàng:', product_id.shape)
print('Lịch sử giá:', product_price_history.shape)

display(product_id.head())
display(product_price_history.head())

## 2. Chuẩn hóa dữ liệu bán tích lũy
Chọn bản ghi mới nhất theo `seller_product_id`, rồi lấy một giá trị đại diện cho mỗi `product_id`. Không xóa trùng theo `product_id` và `quantity_sold`, vì cách đó có thể giữ lại nhiều lần crawl của cùng sản phẩm.

In [ ]:
# Chuyển dữ liệu về kiểu phù hợp
product_price_history['crawl_time'] = pd.to_datetime(
    product_price_history['crawl_time'], errors='coerce'
)
product_price_history['all_time_quantity_sold'] = pd.to_numeric(
    product_price_history['all_time_quantity_sold'], errors='coerce'
)

sales_latest = (
    product_price_history
    .sort_values('crawl_time')
    .drop_duplicates(subset='seller_product_id', keep='last')
    [['product_id', 'seller_product_id', 'all_time_quantity_sold']]
)

# Một product_id có thể có nhiều seller_product_id.
# Dùng giá trị lớn nhất làm đại diện để tránh cộng trùng giữa các gian hàng.
sales_by_product = (
    sales_latest.groupby('product_id', as_index=False)
    ['all_time_quantity_sold'].max()
)

sales_by_product['all_time_quantity_sold'] = (
    sales_by_product['all_time_quantity_sold'].fillna(0)
)

print('Số product_id có dữ liệu bán:', sales_by_product['product_id'].nunique())
display(sales_by_product.head())

## 3. Gắn danh mục cho sản phẩm
Mỗi `product_id` chỉ được tính vào một danh mục trong phép Pareto để tránh đếm trùng. Nếu bảng nguồn có quy tắc xác định danh mục chính, hãy thay `drop_duplicates` bằng quy tắc đó.

In [ ]:
product_category = (
    product_id[['product_id', 'category_id']]
    .dropna(subset=['product_id', 'category_id'])
    .drop_duplicates(subset='product_id', keep='first')
)

category_sales = sales_by_product.merge(
    product_category, on='product_id', how='inner',
    validate='one_to_one'
)

category_sales = category_sales.merge(
    categories[['category_id', 'name']].drop_duplicates('category_id'),
    on='category_id', how='left',
    validate='many_to_one'
)

pareto_df = (
    category_sales.groupby(['category_id', 'name'], as_index=False)
    ['all_time_quantity_sold'].sum()
    .sort_values('all_time_quantity_sold', ascending=False)
    .reset_index(drop=True)
)

display(pareto_df.head())

## 4. Tính Pareto theo danh mục
Tính tỷ trọng và tỷ trọng lũy kế của lượng bán tích lũy. Nhóm được chọn gồm các danh mục cho đến danh mục đầu tiên làm tỷ trọng lũy kế đạt hoặc vượt 80%.

In [ ]:
total_sold = pareto_df['all_time_quantity_sold'].sum()

if total_sold <= 0:
    raise ValueError('Tổng all_time_quantity_sold bằng 0; không thể tính tỷ trọng Pareto.')

pareto_df['sale_ratio'] = (
    pareto_df['all_time_quantity_sold'] / total_sold
)
pareto_df['cumulative_ratio'] = pareto_df['sale_ratio'].cumsum()

# Bao gồm cả danh mục làm tỷ trọng lũy kế vượt ngưỡng 80%
pareto_df['pareto_group'] = 'Nhóm còn lại'
cutoff = pareto_df['cumulative_ratio'].ge(0.80)
if cutoff.any():
    cutoff_index = cutoff.idxmax()
    pareto_df.loc[:cutoff_index, 'pareto_group'] = 'Nhóm đóng góp 80%'

pareto_df['sale_ratio'] = (pareto_df['sale_ratio'] * 100).round(2)
pareto_df['cumulative_ratio'] = (pareto_df['cumulative_ratio'] * 100).round(2)

display(pareto_df)

## 5. Xem kết quả Pareto
Bảng dưới cho biết các danh mục được chọn và tỷ trọng lũy kế của chúng.

In [ ]:
selected_categories = pareto_df[
    pareto_df['pareto_group'] == 'Nhóm đóng góp 80%'
].copy()

print('Số danh mục được chọn:', len(selected_categories))
print('Tỷ trọng lũy kế của nhóm được chọn (%):',
      selected_categories['sale_ratio'].sum().round(2))

display(selected_categories[
    ['category_id', 'name', 'all_time_quantity_sold',
     'sale_ratio', 'cumulative_ratio']
])

## 6. Lấy danh sách sản phẩm cần theo dõi hằng ngày
Giữ nguyên ba cột từ bảng `product_id`: `product_id`, `seller_product_id`, `category_id`. Đây là danh sách đầu vào cho quy trình crawl hằng ngày.

In [ ]:
top_categories = selected_categories['category_id'].unique()

product_id_80 = (
    product_id.loc[
        product_id['category_id'].isin(top_categories),
        ['product_id', 'seller_product_id', 'category_id']
    ]
    .drop_duplicates()
    .reset_index(drop=True)
)

print('Số dòng sản phẩm/gian hàng cần theo dõi:', len(product_id_80))
print('Số product_id duy nhất:', product_id_80['product_id'].nunique())
display(product_id_80.head(10))

## 7. Lưu danh sách theo dõi
File CSV này có thể dùng làm đầu vào cho script crawl giá hằng ngày.

In [ ]:
OUTPUT_FILE = DATA_DIR / 'products' / 'product_id_pareto_80.csv'
product_id_80.to_csv(OUTPUT_FILE, index=False, encoding='utf-8-sig')
print(f'Đã lưu danh sách theo dõi tại: {OUTPUT_FILE}')